# Analyse exploratoire des données — Olist E-Commerce

## Objectif

Cette analyse exploratoire a pour objectif de comprendre la structure,
 les relations entre les six tables retenues du projet Olist
avant leur intégration dans SQL Server et la construction du Data Warehouse.

Les six tables analysées sont :

- Orders
- Order Items
- Products
- Customers
- Reviews
- Sellers

In [2]:
import pandas as pd

In [3]:
orders = pd.read_csv("../dataraw/olist_orders_dataset.csv")
order_items = pd.read_csv("../dataraw/olist_order_items_dataset.csv")
products = pd.read_csv("../dataraw/olist_products_dataset.csv")
customers = pd.read_csv("../dataraw/olist_customers_dataset.csv") 
reviews = pd.read_csv("../dataraw/olist_order_reviews_dataset.csv")
sellers = pd.read_csv("../dataraw/olist_sellers_dataset.csv")


In [4]:
print("Orders :", orders.shape)
print("Order Items :", order_items.shape)
print("Products :", products.shape)
print("Customers :", customers.shape)
print("Reviews :", reviews.shape)
print("Sellers :", sellers.shape)

Orders : (99441, 8)
Order Items : (112650, 7)
Products : (32951, 9)
Customers : (99441, 5)
Reviews : (99224, 7)
Sellers : (3095, 4)


## 1. Compréhension de la structure des tables

Cette étape permet d'identifier les colonnes, les types de données et les
valeurs manquantes de chaque table avant d'analyser les clés et les relations.

##  Table: orders



In [5]:
orders.head(3)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00


In [6]:
orders.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB


##  Table: orders

- Problem 1: Data Types
Convertir a  `datetime`:
* `order_purchase_timestamp`
* `order_approved_at`
* `order_delivered_carrier_date`
* `order_delivered_customer_date`
* `order_estimated_delivery_date`

- Problem 2: Valeur null
* `order_approved_at`: 160 nulls
* `order_delivered_carrier_date`: 1,783 nulls
* `order_delivered_customer_date`: 2,965 nulls

In [10]:
orders["order_id"].nunique()==len(orders)

True

order_id = clé unique de orders
, Grain de orders = 1 ligne représente 1 commande.

-----


##  Table: order_items


In [11]:
order_items.head(3)

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.9,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.9,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.0,17.87


In [12]:
order_items.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  object 
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  object 
 3   seller_id            112650 non-null  object 
 4   shipping_limit_date  112650 non-null  object 
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), object(4)
memory usage: 6.0+ MB


##  Table: orders_items
- Problem 1: Data Types
Convertir a  `datetime`:
* `shipping_limit_date`


In [13]:
print(order_items["order_item_id"].nunique())
print(order_items["order_id"].nunique())
print(order_items.duplicated(
    subset=["order_id", "order_item_id"]
).sum())

21
98666
0


Donc la clé logique de order_items est :(order_id, order_item_id) , Et le grain est :

1 ligne = 1 article d'une commande.

---

##  Table: products


In [14]:
products.head(3)

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0


In [15]:
products.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  object 
 1   product_category_name       32341 non-null  object 
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), object(2)
memory usage: 2.3+ MB


## Table: products

- Problem 1: Data Types
Convertir a  `INT`:
* ` product_name_lenght`
* `product_description_lenght`
* `product_photos_qty`


- Problem 2 : valeur null
* `product_category_name`: 610 nulls
* `product_name_lenght`: 610 nulls
* `product_description_lenght`: 610 nulls
* `product_photos_qty`: 610 nulls
* `product_weight_g`: 2 nulls
* `product_length_cm`: 2 nulls
* `product_height_cm`: 2 nulls
* `product_width_cm`: 2 nulls

In [16]:
products["product_id"].nunique()==len(products)

True

Donc la clé logique de products est product_id , Et le grain est :

1 ligne de products = 1 produit.

----

##  Table: customers


In [17]:
customers.head(3)

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP


In [18]:
customers.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  object
 1   customer_unique_id        99441 non-null  object
 2   customer_zip_code_prefix  99441 non-null  int64 
 3   customer_city             99441 non-null  object
 4   customer_state            99441 non-null  object
dtypes: int64(1), object(4)
memory usage: 3.8+ MB


## Table: customers

- Pas de valeur null
- Pas de Problemes ( Data Types)

In [19]:
customers["customer_id"].nunique()==len(customers)

True

Donc la clé logique de customers est customer_id , Et le grain est :

1 ligne de customers = 1 customer

-----

##  Table: reviews


In [20]:
reviews.head(3)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24


In [21]:
reviews.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   review_id                99224 non-null  object
 1   order_id                 99224 non-null  object
 2   review_score             99224 non-null  int64 
 3   review_comment_title     11568 non-null  object
 4   review_comment_message   40977 non-null  object
 5   review_creation_date     99224 non-null  object
 6   review_answer_timestamp  99224 non-null  object
dtypes: int64(1), object(6)
memory usage: 5.3+ MB


##  Table: reviews

- Problem 1: Data Types
Convertire  a `datetime`:
* `review_creation_date`
* `review_answer_timestamp`

**Problem 2: valeur null
* `review_comment_title`: 87,656 nulls
* `review_comment_message`: 58,247 nulls

Les NULL dans les commentaires ne sont pas forcément un problème. Un client peut simplement laisser une note sans écrire de commentaire.

In [22]:
reviews["review_id"].nunique()

98410

In [23]:
reviews["order_id"].duplicated().sum()

np.int64(551)

In [24]:
int(reviews.duplicated(
    subset=["review_id", "order_id"]
).sum())

0

Clé de reviews :
(review_id, order_id)

Et le grain :

1 ligne = 1 avis (review_id) associé à 1 commande (order_id).

---

##  Table: sellers



In [25]:
sellers.head(3)

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ


In [26]:
sellers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3095 entries, 0 to 3094
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   seller_id               3095 non-null   object
 1   seller_zip_code_prefix  3095 non-null   int64 
 2   seller_city             3095 non-null   object
 3   seller_state            3095 non-null   object
dtypes: int64(1), object(3)
memory usage: 96.8+ KB


## Table: sellers

- Pas de valeur null
- Pas de Problemes ( Data Types)

In [27]:
sellers["seller_id"].nunique()

3095

Clé : seller_id

Grain : 1 ligne = 1 vendeur

-----

# Étape 2 — Vérification des relations entre les tables

<img src="../docs/2_dataset_Olise_de_ce_project.png" width="1000" alt="2_dataset_Olise_de_ce_project.png">

## relation entre orders et order_items

In [28]:
order_items["order_id"].isin(orders["order_id"]).all()

np.True_

tous les `order_id` présents dans order_items existent aussi dans orders.

La relation entre les deux tables est donc cohérente.


## relation entre products et order_items

In [29]:
order_items["product_id"].isin(products["product_id"]).all()

np.True_



Tous les `product_id` présents dans order_items existent dans products.

La relation entre les deux tables est donc cohérente.



## relation entre sellers et order_items

In [30]:
order_items["seller_id"].isin(sellers["seller_id"]).all()

np.True_



Tous les `seller_id` présents dans order_items existent dans sellers.

La relation entre les deux tables est donc cohérente.



## relation entre customers et orders

In [31]:
orders["customer_id"].isin(customers["customer_id"]).all()

np.True_



Tous les `customer_id` présents dans orders existent dans customers.

La relation entre les deux tables est donc cohérente.



## relation entre orders et  reviews

In [32]:
reviews["order_id"].isin(orders["order_id"]).all()

np.True_



Tous les `order_id` présents dans reviews existent dans orders.

La relation entre les deux tables est donc cohérente.



### Synthèse

Les principales relations entre les tables ont été vérifiées à l'aide des
identifiants de référence.
